# TimeXer Experiment


## Dependency


In [45]:
import sys
import importlib.metadata as metadata
print('Python:', sys.executable)
print('Installed NeuralForecast:', metadata.version('neuralforecast'))


Python: c:\ProgramData\anaconda3\envs\timexer\python.exe
Installed NeuralForecast: 3.2.0


## Imports


In [46]:
import os, sys, warnings
import zipfile
from pathlib import Path
warnings.filterwarnings('ignore')
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
import numpy as np
import pandas as pd
import torch
import mlflow
import mlflow.pyfunc
import dagshub
from neuralforecast import NeuralForecast
from neuralforecast.models import TimeXer
from neuralforecast.losses.pytorch import MAE
from src.wmae import wmae
from src.features import load_raw_data, merge_all, build_all_features, TARGET
SEED = 42
DATA_DIR = PROJECT_ROOT
TEST_HORIZON = 39
DEVICE = 'gpu' if torch.cuda.is_available() else 'cpu'
np.random.seed(SEED)
torch.manual_seed(SEED)
print('TimeXer imports successful')
print('Device:', DEVICE)


TimeXer imports successful
Device: gpu


## MLflow Setup


In [47]:
os.environ.setdefault('MLFLOW_TRACKING_USERNAME', 'sansi23')
dagshub.init(repo_owner='sansi23', repo_name='Walmart-Recruiting---Store-Sales-Forecasting', mlflow=True)
timexer_experiment = mlflow.set_experiment('TimeXer_Training')
print('MLflow experiment:', timexer_experiment.name)
print('Tracking URI:', mlflow.get_tracking_uri())


Accessing as sansi23

Initialized MLflow to track repo "sansi23/Walmart-Recruiting---Store-Sales-Forecasting"

Repository sansi23/Walmart-Recruiting---Store-Sales-Forecasting initialized!

2026/07/11 14:52:21 INFO mlflow.tracking.fluent: Experiment with name 'TimeXer_Training' does not exist. Creating a new experiment.


MLflow experiment: TimeXer_Training
Tracking URI: https://dagshub.com/sansi23/Walmart-Recruiting---Store-Sales-Forecasting.mlflow


## Data Preparation


In [48]:
train_raw, test_raw, features_raw, stores_raw = load_raw_data(DATA_DIR)
train_raw, test_raw = merge_all(train_raw, test_raw, features_raw, stores_raw)
all_raw = pd.concat([train_raw.assign(_is_train=1), test_raw.assign(_is_train=0)], ignore_index=True)
all_fe = build_all_features(all_raw, add_lags=False)
all_fe['Weekly_Sales'] = all_fe[TARGET].clip(lower=0)
all_fe['unique_id'] = all_fe['Store'].astype(str) + '_' + all_fe['Dept'].astype(str)
all_fe['ds'] = pd.to_datetime(all_fe['Date'])
FUTURE_EXOG = [
    'Year', 'Month', 'WeekOfYear', 'Quarter',
    'IsSuperBowl', 'IsLaborDay', 'IsThanksgiving', 'IsChristmas', 'IsHoliday',
    'WeeksBefore_SuperBowl', 'WeeksBefore_Thanksgiving', 'WeeksBefore_Christmas',
    'WeeksAfter_SuperBowl', 'WeeksAfter_Thanksgiving', 'WeeksAfter_Christmas',
    'MarkDown_Total', 'MarkDown_Count', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment'
]
FUTURE_EXOG = [c for c in FUTURE_EXOG if c in all_fe.columns]
STATIC_EXOG = ['Store', 'Dept', 'Type_Enc', 'Size']
train_nf = all_fe[all_fe['_is_train'].eq(1)][['unique_id', 'ds', 'Weekly_Sales'] + FUTURE_EXOG].rename(columns={'Weekly_Sales': 'y'}).copy()
test_nf = all_fe[all_fe['_is_train'].eq(0)][['unique_id', 'ds'] + FUTURE_EXOG].copy()
train_nf = train_nf.sort_values(['unique_id', 'ds']).reset_index(drop=True)
test_nf = test_nf.sort_values(['unique_id', 'ds']).reset_index(drop=True)
static_df = all_fe[all_fe['_is_train'].eq(1)][['unique_id', 'Store', 'Dept', 'Type_Enc', 'Size']].drop_duplicates('unique_id').copy()
origin_date = train_nf['ds'].max() - pd.Timedelta(weeks=TEST_HORIZON)
history_counts = train_nf[train_nf['ds'] <= origin_date].groupby('unique_id').size()
future_counts = train_nf[train_nf['ds'] > origin_date].groupby('unique_id').size()
valid_history_ids = set(history_counts[history_counts >= 52].index)
valid_future_ids = set(future_counts[future_counts >= TEST_HORIZON].index)
valid_ids = valid_history_ids.intersection(valid_future_ids)
train_nf = train_nf[train_nf['unique_id'].isin(valid_ids)].copy()
test_nf = test_nf[test_nf['unique_id'].isin(valid_ids)].copy()
static_df = static_df[static_df['unique_id'].isin(valid_ids)].copy()
static_df[['Store', 'Dept', 'Type_Enc', 'Size']] = static_df[['Store', 'Dept', 'Type_Enc', 'Size']].astype(float)
n_series = train_nf['unique_id'].nunique()
print('Train rows:', len(train_nf), '| Test rows:', len(test_nf), '| Series:', n_series)
print('Future exogenous features:', FUTURE_EXOG)


Train rows: 391180 | Test rows: 106817 | Series: 2743
Future exogenous features: ['Year', 'Month', 'WeekOfYear', 'Quarter', 'IsSuperBowl', 'IsLaborDay', 'IsThanksgiving', 'IsChristmas', 'IsHoliday', 'WeeksBefore_SuperBowl', 'WeeksBefore_Thanksgiving', 'WeeksBefore_Christmas', 'WeeksAfter_SuperBowl', 'WeeksAfter_Thanksgiving', 'WeeksAfter_Christmas', 'MarkDown_Total', 'MarkDown_Count', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment']


## Feature Selection


In [49]:
feature_corr = train_nf[FUTURE_EXOG + ['y']].corr(numeric_only=True)['y'].drop('y').abs().sort_values(ascending=False)
selected_exog = feature_corr[feature_corr >= 0.01].index.tolist()
if 'IsHoliday' not in selected_exog:
    selected_exog.append('IsHoliday')
with mlflow.start_run(run_name='TimeXer_Feature_Selection'):
    mlflow.log_params({'candidate_exog_count': len(FUTURE_EXOG), 'selected_exog_count': len(selected_exog), 'selected_exog': ','.join(selected_exog), 'selection_rule': 'absolute Pearson correlation >= 0.01 plus IsHoliday'})
    mlflow.log_metrics({f'corr_{c}': float(feature_corr[c]) for c in feature_corr.index})
print('Selected exogenous features:', selected_exog)


🏃 View run TimeXer_Feature_Selection at: https://dagshub.com/sansi23/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/6/runs/8718c1549c6449bcb9218e1ee8a6fbe0
🧪 View experiment at: https://dagshub.com/sansi23/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/6
Selected exogenous features: ['MarkDown_Total', 'WeeksBefore_Christmas', 'IsThanksgiving', 'Month', 'WeekOfYear', 'Unemployment', 'CPI', 'Quarter', 'MarkDown_Count', 'WeeksBefore_SuperBowl', 'IsHoliday', 'WeeksAfter_Christmas', 'WeeksAfter_Thanksgiving']


## Time-Based Validation


In [50]:
VAL_CUTOFF = train_nf['ds'].max() - pd.Timedelta(weeks=TEST_HORIZON)
tr_nf = train_nf[train_nf['ds'] <= VAL_CUTOFF].copy()
val_nf = train_nf[train_nf['ds'] > VAL_CUTOFF].copy()
val_dates = val_nf[['unique_id', 'ds']].copy()
val_truth = val_nf[['unique_id', 'ds', 'y']].copy()
print('Validation cutoff:', VAL_CUTOFF)
print('Validation weeks:', val_nf['ds'].nunique())


Validation cutoff: 2012-01-27 00:00:00
Validation weeks: 39


## Model Helpers


In [51]:
def build_timexer(config, exog_cols, series_count):
    return TimeXer(
        h=TEST_HORIZON,
        input_size=int(config['input_size']),
        n_series=series_count,
        hist_exog_list=exog_cols,
        stat_exog_list=STATIC_EXOG,
        patch_len=int(config['patch_len']),
        hidden_size=int(config['hidden_size']),
        n_heads=int(config['n_heads']),
        e_layers=int(config['e_layers']),
        d_ff=int(config['d_ff']),
        dropout=float(config['dropout']),
        use_norm=True,
        loss=MAE(),
        valid_loss=MAE(),
        max_steps=int(config['max_steps']),
        learning_rate=float(config['learning_rate']),
        batch_size=int(config['batch_size']),
        windows_batch_size=int(config['windows_batch_size']),
        early_stop_patience_steps=int(config['early_stop_patience_steps']),
        val_check_steps=int(config['val_check_steps']),
        scaler_type='standard',
        random_seed=SEED,
        accelerator=DEVICE,
        devices=1,
        enable_checkpointing=False,
        logger=False,
    )

def evaluate_timexer_forecast(forecast_df, truth_df):
    pred_col = 'TimeXer'
    merged = truth_df.merge(forecast_df[['unique_id', 'ds', pred_col]], on=['unique_id', 'ds'], how='left')
    merged[pred_col] = merged[pred_col].fillna(0.0).clip(lower=0.0)
    holiday_lookup = val_nf[['unique_id', 'ds', 'IsHoliday']].drop_duplicates(['unique_id', 'ds'])
    merged = merged.merge(holiday_lookup, on=['unique_id', 'ds'], how='left')
    score = wmae(merged['y'].to_numpy(), merged[pred_col].to_numpy(), merged['IsHoliday'].fillna(False).to_numpy())
    holiday_mask = merged['IsHoliday'].fillna(False).astype(bool)
    holiday_mae = np.mean(np.abs(merged.loc[holiday_mask, 'y'] - merged.loc[holiday_mask, pred_col])) if holiday_mask.any() else np.nan
    nonholiday_mae = np.mean(np.abs(merged.loc[~holiday_mask, 'y'] - merged.loc[~holiday_mask, pred_col])) if (~holiday_mask).any() else np.nan
    return float(score), float(holiday_mae), float(nonholiday_mae)


## Cross Validation


In [52]:
BASE_CONFIG = {
    'input_size': 52,
    'patch_len': 13,
    'hidden_size': 64,
    'n_heads': 4,
    'e_layers': 2,
    'd_ff': 128,
    'dropout': 0.1,
    'max_steps': 300,
    'learning_rate': 1e-3,
    'batch_size': 8,
    'windows_batch_size': 8,
    'early_stop_patience_steps': -1,
    'val_check_steps': 50,
}
model_cv = build_timexer(BASE_CONFIG, selected_exog, n_series)
nf_cv = NeuralForecast(models=[model_cv], freq='W-FRI')
nf_cv.fit(df=tr_nf[['unique_id', 'ds', 'y'] + selected_exog], static_df=static_df, val_size=0)
fcst_cv = nf_cv.predict()
cv_wmae, cv_mae_holiday, cv_mae_nonholiday = evaluate_timexer_forecast(fcst_cv, val_truth)
with mlflow.start_run(run_name='TimeXer_CV'):
    mlflow.log_params({**BASE_CONFIG, 'validation_weeks': TEST_HORIZON, 'n_series': n_series, 'selected_exog': ','.join(selected_exog), 'metric': 'WMAE_raw_scale'})
    mlflow.log_metrics({'cv_wmae': cv_wmae, 'cv_mae_holiday': cv_mae_holiday, 'cv_mae_nonholiday': cv_mae_nonholiday})
print(f'CV WMAE: {cv_wmae:,.2f}')
print(f'Holiday MAE: {cv_mae_holiday:,.2f} | Non-Holiday MAE: {cv_mae_nonholiday:,.2f}')


Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

   | Name                | Type                   | Params | Mode 
------------------------------------------------------------------------
0  | loss                | MAE                    | 0      | train
1  | valid_loss          | MAE                    | 0      | train
2  | hist_cat_embeddings | ModuleList             | 0      | train
3  | futr_cat_embeddings | ModuleList             | 0      | train
4  | stat_cat_embeddings | ModuleList             | 0      | train
5  | padder_train        | ConstantPad1d          | 0      | train
6  | scaler              | TemporalNorm           | 0      | train
7  | en_embedding        | EnEmbedding            | 176 K  | train
8  | ex_embedding        | DataEmbedding_inverted | 3.4 K  | train
9  | encoder             | Encoder                | 100 K  | train
10 | head                | FlattenHead            | 

Epoch 299: 100%|██████████| 1/1 [00:01<00:00,  0.95it/s, train_loss_step=0.678, train_loss_epoch=0.678]

`Trainer.fit` stopped: `max_steps=300` reached.


Epoch 299: 100%|██████████| 1/1 [00:01<00:00,  0.95it/s, train_loss_step=0.678, train_loss_epoch=0.678]

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]



Predicting DataLoader 0: 100%|██████████| 1/1 [00:00<00:00, 29.64it/s] 
🏃 View run TimeXer_CV at: https://dagshub.com/sansi23/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/6/runs/34c85184224347bab9c95177cc3bbb4b
🧪 View experiment at: https://dagshub.com/sansi23/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/6
CV WMAE: 1,785.06
Holiday MAE: 1,982.30 | Non-Holiday MAE: 1,731.76


## Hyperparameter Tuning


In [53]:
timexer_search_space = [
    {**BASE_CONFIG, 'patch_len': 13, 'hidden_size': 64, 'learning_rate': 1e-3},
    {**BASE_CONFIG, 'patch_len': 26, 'hidden_size': 64, 'learning_rate': 1e-3},
    {**BASE_CONFIG, 'patch_len': 13, 'hidden_size': 128, 'learning_rate': 5e-4},
    {**BASE_CONFIG, 'patch_len': 26, 'hidden_size': 128, 'learning_rate': 5e-4},
]
timexer_results = []
for trial_id, config in enumerate(timexer_search_space, start=1):
    model_trial = build_timexer(config, selected_exog, n_series)
    nf_trial = NeuralForecast(models=[model_trial], freq='W-FRI')
    nf_trial.fit(df=tr_nf[['unique_id', 'ds', 'y'] + selected_exog], static_df=static_df, val_size=0)
    fcst_trial = nf_trial.predict()
    score, holiday_mae, nonholiday_mae = evaluate_timexer_forecast(fcst_trial, val_truth)
    timexer_results.append({'trial_id': trial_id, **config, 'wmae': score, 'mae_holiday': holiday_mae, 'mae_nonholiday': nonholiday_mae})
    with mlflow.start_run(run_name=f'TimeXer_trial_{trial_id:02d}', nested=True):
        mlflow.log_params({**config, 'validation_weeks': TEST_HORIZON, 'n_series': n_series, 'selected_exog': ','.join(selected_exog), 'metric': 'WMAE_raw_scale'})
        mlflow.log_metrics({'validation_wmae': score, 'validation_mae_holiday': holiday_mae, 'validation_mae_nonholiday': nonholiday_mae})
    print(f'Trial {trial_id}: WMAE={score:,.2f}')
timexer_tuning_df = pd.DataFrame(timexer_results).sort_values('wmae').reset_index(drop=True)
best_timexer = timexer_tuning_df.iloc[0].to_dict()
best_timexer_config = {k: best_timexer[k] for k in BASE_CONFIG}
timexer_tuning_df.to_csv('timexer_tuning_results.csv', index=False)
display(timexer_tuning_df)


Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

   | Name                | Type                   | Params | Mode 
------------------------------------------------------------------------
0  | loss                | MAE                    | 0      | train
1  | valid_loss          | MAE                    | 0      | train
2  | hist_cat_embeddings | ModuleList             | 0      | train
3  | futr_cat_embeddings | ModuleList             | 0      | train
4  | stat_cat_embeddings | ModuleList             | 0      | train
5  | padder_train        | ConstantPad1d          | 0      | train
6  | scaler              | TemporalNorm           | 0      | train
7  | en_embedding        | EnEmbedding            | 176 K  | train
8  | ex_embedding        | DataEmbedding_inverted | 3.4 K  | train
9  | encoder             | Encoder                | 100 K  | train
10 | head                | FlattenHead            | 

Epoch 299: 100%|██████████| 1/1 [00:01<00:00,  0.93it/s, train_loss_step=0.681, train_loss_epoch=0.681]

`Trainer.fit` stopped: `max_steps=300` reached.


Epoch 299: 100%|██████████| 1/1 [00:01<00:00,  0.93it/s, train_loss_step=0.681, train_loss_epoch=0.681]

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


...

`Trainer.fit` stopped: `max_steps=300` reached.


Epoch 299: 100%|██████████| 1/1 [00:01<00:00,  0.77it/s, train_loss_step=0.696, train_loss_epoch=0.696]


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting DataLoader 0: 100%|██████████| 1/1 [00:00<00:00, 26.64it/s]
🏃 View run TimeXer_trial_04 at: https://dagshub.com/sansi23/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/6/runs/d4492192db9b4e719c39c8cf20970aa4
🧪 View experiment at: https://dagshub.com/sansi23/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/6
Trial 4: WMAE=1,828.96


,trial_id,input_size,patch_len,hidden_size,n_heads,e_layers,d_ff,dropout,max_steps,learning_rate,batch_size,windows_batch_size,early_stop_patience_steps,val_check_steps,wmae,mae_holiday,mae_nonholiday
0,1,52,13,64,4,2,128,0.1,300,0.0010,8,8,-1,50,1788.035956,1981.196710,1735.830347
1,4,52,26,128,4,2,128,0.1,300,0.0005,8,8,-1,50,1828.956992,2021.651881,1776.877292
2,2,52,26,64,4,2,128,0.1,300,0.0010,8,8,-1,50,1832.207938,2060.916161,1770.394905
3,3,52,13,128,4,2,128,0.1,300,0.0005,8,8,-1,50,1852.993491,2215.908244,1754.908422


## Final Pipeline and Model Registry


In [54]:
class TimeXerInferencePipeline(mlflow.pyfunc.PythonModel):
    def __init__(self, neural_forecast, exog_cols, static_df):
        self.neural_forecast = neural_forecast
        self.exog_cols = list(exog_cols)
        self.static_df = static_df.copy()

    def _prepare(self, raw):
        raw = raw.copy().reset_index(drop=True)
        raw['_input_order'] = np.arange(len(raw))
        frame = build_all_features(raw, add_lags=False)
        frame['unique_id'] = frame['Store'].astype(str) + '_' + frame['Dept'].astype(str)
        frame['ds'] = pd.to_datetime(frame['Date'])
        return frame[['unique_id', 'ds', '_input_order'] + self.exog_cols].sort_values(['unique_id', 'ds'])

    def predict(self, context, model_input, params=None):
        future = self._prepare(model_input)
        input_order = future[['unique_id', 'ds', '_input_order']].copy()
        forecast = self.neural_forecast.predict()
        forecast = forecast.rename(columns={'TimeXer': 'Weekly_Sales'})
        output = input_order.merge(forecast[['unique_id', 'ds', 'Weekly_Sales']], on=['unique_id', 'ds'], how='left')
        return output.sort_values('_input_order')[['Weekly_Sales']].fillna(0.0).clip(lower=0.0).reset_index(drop=True)

final_model = build_timexer(best_timexer_config, selected_exog, n_series)
nf_final = NeuralForecast(models=[final_model], freq='W-FRI')
nf_final.fit(df=train_nf[['unique_id', 'ds', 'y'] + selected_exog], static_df=static_df, val_size=0)
timexer_pipeline = TimeXerInferencePipeline(nf_final, selected_exog, static_df)
pipeline_predictions = timexer_pipeline.predict(None, test_raw)['Weekly_Sales'].to_numpy()
assert len(pipeline_predictions) == len(test_raw)
assert np.isfinite(pipeline_predictions).all()
assert (pipeline_predictions >= 0).all()
print('Pipeline check rows:', len(pipeline_predictions))
print('Best configuration:', best_timexer_config)


c:\ProgramData\anaconda3\envs\timexer\Lib\site-packages\mlflow\pyfunc\utils\data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(
Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

   | Name                | Type                   | Params | Mode 
------------------------------------------------------------------------
0  | loss                | MAE                    | 0      | train
1  | valid_loss          | MAE                    | 0      | train
2  | hist_cat_embeddings | ModuleList             | 0      | train
3  | futr_cat_embeddings | ModuleList             | 0      | train
4  | stat_cat_embeddings | ModuleList             | 0      | train
5  | padder_train        |

Epoch 299: 100%|██████████| 1/1 [00:01<00:00,  0.95it/s, train_loss_step=0.637, train_loss_epoch=0.637]

`Trainer.fit` stopped: `max_steps=300` reached.


Epoch 299: 100%|██████████| 1/1 [00:01<00:00,  0.95it/s, train_loss_step=0.637, train_loss_epoch=0.637]


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting DataLoader 0: 100%|██████████| 1/1 [00:00<00:00, 29.00it/s] 
Pipeline check rows: 115064
Best configuration: {'input_size': 52.0, 'patch_len': 13.0, 'hidden_size': 64.0, 'n_heads': 4.0, 'e_layers': 2.0, 'd_ff': 128.0, 'dropout': 0.1, 'max_steps': 300.0, 'learning_rate': 0.001, 'batch_size': 8.0, 'windows_batch_size': 8.0, 'early_stop_patience_steps': -1.0, 'val_check_steps': 50.0}


## MLflow Registration


In [56]:
FINAL_MODEL_NAME = 'WalmartSales_TimeXer'
with mlflow.start_run(run_name='TimeXer_Final') as run:
    mlflow.log_params({
        **{f'final_{k}': v for k, v in best_timexer_config.items()},
        'model_family': 'TimeXer',
        'registered_model_name': FINAL_MODEL_NAME,
        'selection_source': 'TimeXer_Tuning',
        'selected_exog': ','.join(selected_exog),
        'final_train_rows': len(train_nf),
        'test_rows': len(test_nf),
        'validation_weeks': TEST_HORIZON,
        'feature_contract': 'raw_merged_to_weekly_sales',
        'pipeline_preserves_input_order': True,
    })
    mlflow.log_metric('selection_wmae', float(best_timexer['wmae']))
    mlflow.log_artifact('timexer_tuning_results.csv')
    mlflow.set_tags({'pipeline_contract': 'raw_merged_dataframe_to_weekly_sales', 'winner_selected_by': 'minimum_validation_wmae'})
    mlflow.pyfunc.log_model(name='timexer_pipeline', python_model=timexer_pipeline, registered_model_name=FINAL_MODEL_NAME)
print('Run ID:', run.info.run_id)
print('Model registered as:', FINAL_MODEL_NAME)


2026/07/11 15:24:57 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.
Successfully registered model 'WalmartSales_TimeXer'.
2026/07/11 15:26:20 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: WalmartSales_TimeXer, version 1
Created version '1' of model 'WalmartSales_TimeXer'.


🏃 View run TimeXer_Final at: https://dagshub.com/sansi23/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/6/runs/0b8b5aa1488d4393bbeeb636ad758f14
🧪 View experiment at: https://dagshub.com/sansi23/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/6
Run ID: 0b8b5aa1488d4393bbeeb636ad758f14
Model registered as: WalmartSales_TimeXer
